<a href="https://colab.research.google.com/github/jefferyocran/FraudGuard-OXGBoost/blob/main/experiment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -U xgboost -q

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import recall_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
print("Ready.")


Ready.


In [9]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
uci = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
uci['label'] = uci['label'].map({'ham': 0, 'spam': 1})
uci['source'] = 'uci'

field = pd.read_csv('ghana_momo_field_v3.csv')[['label', 'message']]
field['source'] = 'field'

print("UCI:", len(uci), "| Field:", len(field))


UCI: 5572 | Field: 67


In [10]:
field_train, field_test = train_test_split(
    field, test_size=0.4, stratify=field['label'], random_state=SEED
)
print("Field train:", len(field_train), "| Field test:", len(field_test))


Field train: 40 | Field test: 27


In [11]:
train_A = uci.copy()                                    # UCI only
train_B = pd.concat([uci, field_train], ignore_index=True)  # UCI + Ghana

print("Condition A:", len(train_A), "| Condition B:", len(train_B))


Condition A: 5572 | Condition B: 5612


In [12]:
def train_and_test(train_df, test_df, name, field_boost=1):
    vec = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
    X_train = vec.fit_transform(train_df['message'])
    y_train = train_df['label']
    X_test = vec.transform(test_df['message'])
    y_test = test_df['label']

    # Give Ghanaian rows more weight so they aren't drowned by UCI
    weights = np.where(train_df['source'] == 'field', field_boost, 1)

    model = xgb.XGBClassifier(
        max_depth=6, n_estimators=200, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, eval_metric='logloss'
    )
    model.fit(X_train, y_train, sample_weight=weights)
    preds = model.predict(X_test)

    scam_recall = recall_score(y_test, preds, pos_label=1)
    print(f"\n=== {name} ===")
    print(f"Ghanaian SCAM recall: {round(scam_recall*100, 1)}%")
    print(confusion_matrix(y_test, preds))
    return scam_recall

recall_A = train_and_test(train_A, field_test, "Condition A: UCI only", field_boost=1)
recall_B = train_and_test(train_B, field_test, "Condition B: UCI + Ghana (weighted)", field_boost=100)

print("\n----------------------------------------")
print(f"Without local data:       {round(recall_A*100,1)}%")
print(f"With weighted local data: {round(recall_B*100,1)}%")
print(f"Improvement:              {round((recall_B-recall_A)*100,1)} pp")



=== Condition A: UCI only ===
Ghanaian SCAM recall: 29.4%
[[10  0]
 [12  5]]

=== Condition B: UCI + Ghana (weighted) ===
Ghanaian SCAM recall: 52.9%
[[9 1]
 [8 9]]

----------------------------------------
Without local data:       29.4%
With weighted local data: 52.9%
Improvement:              23.5 pp
